# Improved LLM Plans
Original feature, Only one mental Keggle Mental health information used.
Then, based on information, how should I imporve my model,
In inital Training, gpt model4, and we have serveral information inside.
Next approch, I like to Hugging Datasets and Mental health datasets

Currently, feature we just populate randomly.
Next what, I want is text related llm topic clustering,
Based on token, clustering.


In [1]:
# Hugging Datasets called
raw_data_path = "raw_data"
cleaned_data_path= "cleaned_data"



In [2]:
import pandas as pd
# Inital Raw Data Used.

def clean_keggle_df(data_path):
    df = pd.read_csv(data_path)
    df = df[["questionText", "topics", "re_diagnosis","clean_answer_text"]]
    # Lower case
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    # remove non-world
    df = df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    # remove number
    df = df.replace(to_replace=r'\d', value='', regex=True)

    return df

def clean_hugging_df(data_path):
    df = pd.read_csv(data_path)

    df = df[["questionTitle", "questionText", "topic", "answerText"]]
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    df["questionText"] = df["questionTitle"].fillna('') + " " + df["questionText"].fillna('')
    df = df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    df = df.replace(to_replace=r'\d', value='', regex=True)
    df = df[["questionText", "topic", "answerText"]]

    # print(hugging_df.head)
    return df

keggle_df = clean_keggle_df(f"{raw_data_path}/counsel_cleaned.csv")   
print(keggle_df.shape)
keggle_df.to_csv(f"{cleaned_data_path}/cleaned_counsel.csv")
hugging_df = clean_hugging_df(f"{raw_data_path}/huggin_counsel_chat.csv")
print(hugging_df.shape)
hugging_df

(1373, 4)
(2775, 3)


/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_86348/1977113534.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_86348/1977113534.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


,questionText,topic,answerText
0,do i have too many issues for counseling i hav...,depression,it is very common for people to have multiple ...
1,do i have too many issues for counseling i hav...,depression,ive never heard of someone having too many iss...
2,do i have too many issues for counseling i hav...,depression,absolutely not i strongly recommending workin...
3,do i have too many issues for counseling i hav...,depression,let me start by saying there are never too man...
4,do i have too many issues for counseling i hav...,depression,i just want to acknowledge you for the courage...
...,...,...,...
2770,are some clients more difficult than others wh...,counselingfundamentals,although many clients have the capacity to be ...
2771,are some clients more difficult than others wh...,counselingfundamentals,i usually dont label a client as difficult bec...
2772,are some clients more difficult than others wh...,counselingfundamentals,dang right heh heh and correct me if im wrong...
2773,are some clients more difficult than others wh...,counselingfundamentals,yes just like some relationships outside of ou...


In [3]:
import pandas as pd

# Read File Information
hugging_df = pd.read_csv(f"{cleaned_data_path}/cleaned_hugging.csv")
counsel_df = pd.read_csv(f"{cleaned_data_path}/cleaned_counsel.csv")

# Change Column Name
hugging_df.rename(columns={"questionText": "question_text",
                           "topic": "topics", 
                           "answerText": "answer_text"}, inplace=True)
hugging_df.drop(columns=['Unnamed: 0'], inplace=True)

print("After renaming:", hugging_df.columns)
# Drop unused column
counsel_df.drop(columns=['Unnamed: 0', 're_diagnosis'], inplace=True)
print(counsel_df.columns)
counsel_df.rename(columns={"questionText": "question_text", 
                           "clean_answer_text": "answer_text"}, inplace=True)

# Select target column
hugging_df = hugging_df[["question_text", "topics", "answer_text"]]
counsel_df = counsel_df[["question_text", "topics", "answer_text"]]
print(hugging_df.columns)
print(counsel_df.columns)

# Concat Column
combined_dataset = pd.concat([hugging_df, counsel_df], ignore_index=True)

# Combined Output
print(combined_dataset.columns)
combined_dataset.to_csv(f"{cleaned_data_path}/combined_output.csv")


After renaming: Index(['questionTitle', 'question_text', 'topics', 'answer_text'], dtype='object')
Index(['questionText', 'topics', 'clean_answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')


In [4]:
# Chat Promt, Design.
combined_dataset = pd.read_csv(f"{cleaned_data_path}/combined_output.csv")
print(combined_dataset.columns)



Index(['Unnamed: 0', 'question_text', 'topics', 'answer_text'], dtype='object')


In [5]:
# NLTK test
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
text = "This is an example. Here is another sentence."
sentences = sent_tokenize(text)

print(sentences)


['This is an example.', 'Here is another sentence.']


[nltk_data] Downloading package punkt_tab to /Users/yoon/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
# Cleaned Combined Datasets too shorts and too long
import re
from nltk.tokenize import word_tokenize

def is_noisy(text: str) -> bool:
    if re.search(r'[가-힣A-Za-z]', text) is None:
        return True
    cleaned = re.sub(r'[^\w\s]', '', text) 
    if len(cleaned) == 0 or len(cleaned) < len(text) * 0.02:  
        return True
    return False

def clean_combined_dataset(df):
    def token_count(text):
        tokens = sent_tokenize(text)
        return len(tokens)
    # Apply the token_count function to calculate the number of tokens in questions and answers
    df = df.dropna()
    df['q_token_count'] = df['question_text'].apply(token_count)
    df['a_token_count'] = df['answer_text'].apply(token_count)
    
    return df
# print(combined_dataset.columns)
print(f"Before: {combined_dataset.shape}")
cleaned_combined_df = clean_combined_dataset(combined_dataset)
cleaned_combined_df.to_csv(f"{cleaned_data_path}/cleaned_combined.csv")
print(f"After: {cleaned_combined_df.shape}")


Before: (4148, 4)
After: (3985, 6)


/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_86348/339852553.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['q_token_count'] = df['question_text'].apply(token_count)
/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_86348/339852553.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['a_token_count'] = df['answer_text'].apply(token_count)


In [7]:
# Lamma testing Device
from datasets import Dataset, DatasetDict

target_df = pd.read_csv(f"{cleaned_data_path}/cleaned_combined.csv")
print(f"After: {target_df.shape}")
print(f"Columns: {target_df.columns}")

system_prompt = (
    "You are a mental health assistant trained in cognitive behavioral therapy. "
    "Help users explore their thoughts and feelings with empathy and guidance."
)

def format_to_chat_messages(example):
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": example["question_text"] + f" based on {example['topics']}"},
            {"role": "assistant", "content": example["answer_text"]},
        ]
    }

# Grab only nessary columns only
target_df = target_df[["question_text", "topics", "answer_text"]].dropna()
hf_dataset = Dataset.from_pandas(target_df)

# Hugging Face Datasets
hf_dataset = hf_dataset.map(format_to_chat_messages)
hf_dataset = hf_dataset.remove_columns([col for col in hf_dataset.column_names if col != "messages"])
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)

train_data_path = "train_data"
hf_dataset["train"].to_json(f"{train_data_path}/train_dataset.json", orient="records", force_ascii=False)
hf_dataset["test"].to_json(f"{train_data_path}/test_dataset.json", orient="records", force_ascii=False)
print(hf_dataset["train"][0])
print(hf_dataset["test"][0])



/Users/yoon/Desktop/GeorgiaTech-Assingment/Mental-Health-AI-Driven-System-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


After: (3985, 7)
Columns: Index(['Unnamed: 0.1', 'Unnamed: 0', 'question_text', 'topics', 'answer_text',
       'q_token_count', 'a_token_count'],
      dtype='object')


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 158.75ba/s]

{'messages': [{'content': "This is a Mental Health ChatBot assistant designed based on actual consultation data and implemented using real test cases.Its purpose is to accurately understand users' questions and respond with comforting and helpful messages.The focus is primarily on the question_text, and when necessary, it can provide various empathetic expressions. In cases where the user's question is unclear or emotionally unstable, the assistant should request additional clarification, while also offering basic empathy and supportive advice", 'role': 'system'}, {'content': 'ive been experiencing a lot of anxiety and panic attacks lately i was recently diagnosed by my psychiatrist with obsessivecompulsive disorder lately ive been questioning everything from my career to my relationship my boyfriend and i just moved in a few months ago all of a sudden i dont feel as comfortable around him as i used to although i cant seem to find a reason as to why i feel this way based on anxiety', '

In [8]:
import os
import random
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, set_seed
from trl import SFTTrainer

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"  # or Falcon-RW-1B
DATA_DIR = "./train_data"
OUTPUT_DIR = "./llama3-mental-health"
MAX_SEQ_LENGTH = 512

# === Chat Template (LLaMA3 스타일) ===
LLAMA_3_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}{{ message['content'] }}"
    "{% elif message['role'] == 'user' %}{{ '\n\nHuman: ' + message['content'] + eos_token }}"
    "{% elif message['role'] == 'assistant' %}{{ '\n\nAssistant: ' + message['content'] + eos_token }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '\n\nAssistant: ' }}{% endif %}"
)

# === Step 1: 데이터 로드 ===
train_df = pd.read_json(f"{DATA_DIR}/train_dataset.json", lines = True)
test_df = pd.read_json(f"{DATA_DIR}/test_dataset.json", lines = True)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = LLAMA_3_CHAT_TEMPLATE.replace("{{ eos_token }}", tokenizer.eos_token)
tokenizer.padding_side = "right"

def convert(example):
    prompt = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return tokenizer(prompt, truncation=True, padding="max_length", max_length=MAX_SEQ_LENGTH)

train_dataset = train_dataset.map(convert, remove_columns=["messages"])
test_dataset = test_dataset.map(convert, remove_columns=["messages"])

# === Step 4: 모델 로드 ===
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# === Step 5: 학습 설정 ===
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    save_strategy="epoch",
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none",
)

# === Step 6: SFTTrainer 설정 ===
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=training_args,
    dataset_text_field="input_ids",
    packing=False
)

# === Step 7: 학습 실행 ===
trainer.train()

# === Step 8: 저장 ===
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 모델 저장 완료: {OUTPUT_DIR}")


/Users/yoon/Desktop/GeorgiaTech-Assingment/Mental-Health-AI-Driven-System-Project/.venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
403 Client Error. (Request ID: Root=1-6806a24e-75204b7e4682aeac7c2ec2ff;0d55639b-df36-407b-ac7e-4190b12ca623)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct to ask for access.

In [8]:
import pandas as pd
df = pd.read_csv("cleaned_data/cleaned_combined.csv")

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("nmcahill/mbti-classifier")
model = AutoModelForSequenceClassification.from_pretrained("nmcahill/mbti-classifier")

c:\Users\ykim\Desktop\Personal\Mental-Health-AI-Driven-System-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ykim\Desktop\Personal\Mental-Health-AI-Driven-System-Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ykim\.cache\huggingface\hub\models--nmcahill--mbti-classifier. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activ

In [ ]:
import torch
import torch.nn.functional as F

df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])
df_unique_questions = df.drop_duplicates(subset='question_text')

df_sampled = df_unique_questions.sample(n=300, random_state = 42)
print(df.columns)


text = "I enjoy helping others and often feel emotionally connected to people."
inputs = tokenizer(text, return_tensors="pt")



def classify_tf(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    # Get model output
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_idx = torch.argmax(F.softmax(logits, dim=1), dim=1).item()
    label = mbti_labels[pred_idx]
    return "T" if "T" in label else "F"

# Load MBTI label names
mbti_labels = [
    "INFJ", "INFP", "INTJ", "INTP", "ISFJ", "ISFP", "ISTJ", "ISTP",
    "ENFJ", "ENFP", "ENTJ", "ENTP", "ESFJ", "ESFP", "ESTJ", "ESTP"
]

# Get top predicted label
df_sampled['tf_type'] = df_sampled['answer_text'].apply(classify_tf)


df


Index(['question_text', 'topics', 'answer_text', 'q_token_count',
       'a_token_count', 'tf_type'],
      dtype='object')


,question_text,topics,answer_text,q_token_count,a_token_count,tf_type
1379,i snap easy and push people away i need help b...,angermanagement,death of someone with whom we had fond involve...,1,1,T
1620,my boyfriend of five years told me he cheated ...,intimacy,hi michiganthis is a common issue how do you t...,1,1,T
1295,i was the one who ended it and im so glad i di...,trauma,ending an abusive relationship is often very d...,1,1,T
422,a girl and i were madly in love we dated for o...,depression,hi boise im sorry that youve lost this love th...,1,1,T
1364,when i got home my boyfriend and i got into an...,angermanagement,sounds scary to watch i agree with youmaybe h...,1,1,T


In [18]:
df_t = df_sampled[df_sampled["tf_type"] == "T"]
print(df_t.shape)
df_f = df_sampled[df_sampled["tf_type"] == "F"]
print(df_f.shape)

(300, 6)
(0, 6)


In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "datasnaek/mbti-type",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

In [19]:
df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])
df_unique_questions = df.drop_duplicates(subset='question_text')

df_sampled = df_unique_questions.sample(n=300, random_state = 42)
print(df.columns)


from transformers import pipeline
import pandas as pd
from tqdm import tqdm

# Zero-shot classification pipeline (BART 기반)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 예: 샘플 데이터프레임 (여기에 실제 df_sampled 사용)
# df_sampled = pd.read_csv("your_answer_text_sample.csv")  # 만약 외부에서 로딩할 경우
# 여기선 answer_text 컬럼이 있다고 가정

# Zero-shot 기반 T/F 분류 함수
def classify_tf_zero_shot(text):
    labels = ["logical reasoning", "emotional support"]
    try:
        result = classifier(text, candidate_labels=labels)
        top_label = result['labels'][0]
        score = result['scores'][0]
        tf_type = "T" if top_label == "logical reasoning" else "F"
        return pd.Series([tf_type, score])
    except:
        return pd.Series([None, None])  # 오류 방지

# tqdm으로 진행 상황 보이기 + 결과 저장
tqdm.pandas()
df_sampled[['tf_type_zero_shot', 'tf_score']] = df_sampled['answer_text'].progress_apply(classify_tf_zero_shot)


Index(['question_text', 'topics', 'answer_text', 'q_token_count',
       'a_token_count', 'tf_type'],
      dtype='object')


c:\Users\ykim\Desktop\Personal\Mental-Health-AI-Driven-System-Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ykim\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not

In [25]:
df_sampled.to_csv("cleaned_data/sample.csv")
df_sampled_T = df_sampled[df_sampled['tf_type_zero_shot'] == "F"]
print(df_sampled_T.shape)
print(df_sampled_T)
df_sampled_F = df_sampled[df_sampled['tf_type_zero_shot'] == "T"]
print(df_sampled_F.shape)


(239, 8)
                                          question_text  \
1379  i snap easy and push people away i need help b...   
1620  my boyfriend of five years told me he cheated ...   
1295  i was the one who ended it and im so glad i di...   
422   a girl and i were madly in love we dated for o...   
1364  when i got home my boyfriend and i got into an...   
...                                                 ...   
2680  hes been losing feelings and he doesnt know wh...   
2745  in  my former husband of  years walked away fr...   
1631  over the course of a few days my wife was unsu...   
434   im unemployed just relocated i cant get approv...   
3141  i feel like every time i do something someone ...   

                                 topics  \
1379                    angermanagement   
1620                           intimacy   
1295                             trauma   
422                          depression   
1364                    angermanagement   
...                     